In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/УСЛОВИЯ.md
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/~$СЛОВИЯ.docx
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/УСЛОВИЯ.pdf
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/УСЛОВИЯ.docx
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/style_clf.pkl
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/adapter_model.safetensors
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/merges.txt
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/value_head.safetensors
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/adapter_config.json
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/tokenizer.json
/kaggle/input

In [2]:
import json
base = '/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/data'

def peek(name, n=2):
    print("="*70); print(name); print("="*70)
    with open(f'{base}/{name}.jsonl') as f:
        for i, line in enumerate(f):
            if i >= n: break
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
            print("-"*40)

for name in ['kid_adult','good_bad','public_test_style','public_test_quality']:
    with open(f'{base}/{name}.jsonl') as f:
        print(name, '—', sum(1 for _ in f), 'записей')
print()
peek('kid_adult')
peek('public_test_style')

kid_adult — 1489 записей
good_bad — 2226 записей
public_test_style — 50 записей
public_test_quality — 50 записей

kid_adult
{
  "prompt": "Как работает камера видеонаблюдения? Ответ: Она ловит свет через стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.",
  "kid": "Камера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и сохраняет как видео в памяти.",
  "adult": "Свет проходит через объектив на светочувствительную матрицу, которая преобразует фотоны в электрические сигналы. Затем процессор кодирует эти данные в цифровой формат и сохраняет итоговый видеопоток на накопитель."
}
----------------------------------------
{
  "prompt": "Что такое химическая реакция? Это процесс, при котором одни вещества превращаются в совершенно другие.",
  "kid": "Это когда вещества встречаются и превращаются во что-то совсем новое, как когда из сырых продуктов в духовке получается вкусный пирог. Настоящее волш

In [3]:
!pip install -q "transformers==4.53.2" "huggingface_hub==0.33.2" "trl==0.19.1" "peft==0.16.0" "accelerate==1.8.1" "bitsandbytes==0.46.1" "datasets==3.6.0"


In [4]:
import torch, random, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL = "Qwen/Qwen3-4B-Instruct-2507"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16,
)
print("модель:", model.config.model_type, "| слоёв:", model.config.num_hidden_layers)
print("память GPU занята:", round(torch.cuda.memory_allocated()/1e9, 2), "ГБ")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

модель: qwen3 | слоёв: 36
память GPU занята: 1.15 ГБ


In [6]:
import json

base_data = '/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/data'
tests = [json.loads(l) for l in open(f'{base_data}/public_test_style.jsonl')]

def generate(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

print("ВОПРОС:\n", tests[0]["prompt"][:150])
print("\nОТВЕТ БАЗОВОЙ МОДЕЛИ (до SFT):\n", generate(tests[0]["prompt"]))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ВОПРОС:
 Почему продавцы в магазинах делают скидки и распродажи? Ответ: Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привле

ОТВЕТ БАЗОВОЙ МОДЕЛИ (до SFT):
 Правильный ответ: **Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.**

Это объяснение корректно и полно. Продавцы используют скидки и распродажи для нескольких ключевых целей:

1. **Ускорения продаж старых товаров** — товары, которые уже не востребованы, могут быть проданы дешевле, чтобы избежать их скопления.
2. **Освобождения места** — чтобы освободить витрины и складские площади для новых, более актуальных или модных товаров.
3. **Привлечения покупателей** — скидки создают чувство выгоды, стимулируют покупателей прийти в магазин, увеличивают трафик и могут привести к росту общего дохода.

Таким образом, ответ — **правильный и полный**. ✅


In [7]:
from datasets import Dataset
base_data = base  # тот же путь
rows = [json.loads(l) for l in open(f'{base_data}/kid_adult.jsonl')]
train = Dataset.from_list([
    {"messages": [
        {"role": "user", "content": r["prompt"]},
        {"role": "assistant", "content": r["kid"]},
    ]}
    for r in rows
])
print(train)
print(train[0]["messages"])

Dataset({
    features: ['messages'],
    num_rows: 1489
})
[{'role': 'user', 'content': 'Как работает камера видеонаблюдения? Ответ: Она ловит свет через стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.'}, {'role': 'assistant', 'content': 'Камера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и сохраняет как видео в памяти.'}]


In [8]:
from peft import LoraConfig
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
print(lora)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'o_proj', 'k_proj', 'up_proj', 'v_proj', 'down_proj', 'q_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)


In [10]:
from trl import SFTConfig, SFTTrainer
cfg = SFTConfig(
    output_dir="sft_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    max_length=1024,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=42,
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=train,
    peft_config=lora,
    processing_class=tokenizer,
)
trainer.train()

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.999200
20,1.397700
30,1.304600
40,1.287400
50,1.293100
60,1.231800
70,1.204100
80,1.187100
90,1.200700
100,1.181200


TrainOutput(global_step=187, training_loss=1.244296101962819, metrics={'train_runtime': 789.3861, 'train_samples_per_second': 1.886, 'train_steps_per_second': 0.237, 'total_flos': 4834071741450240.0, 'train_loss': 1.244296101962819})